# Suite RND — Random sources

A store is equipped with a random source when it is made, for whatever draws from it. `Stores.Random` is the protocol
and `Stores.PCG32` the reference source, specified exactly, so that both implementations draw the same numbers.

In [ ]:
from mbse.Schemas.Framework import Bindings as Bd, Proxies, Schemas as S, Stores
from support import conformance_problems, raises, text

REFERENCE = [0xA15C02B7, 0x7B47F409, 0xBA1D3330, 0x83D2F293, 0xBFA4784B, 0xCBED606E]  # pcg32-demo: seed 42, sequence 54

## RND-01 · PCG32 draws the reference sequence

In [ ]:
source = Stores.PCG32(42, 54)
assert [source.next_u32() for _ in range(6)] == REFERENCE and (source.seed, source.sequence) == (42, 54)
assert Stores.PCG32(42).sequence == 0 and Stores.PCG32(2**64 - 1, 2**64 - 1).next_u32() < 2**32
for seed, sequence in [(-1, 0), (2**64, 0), (0, -1), (0, 2**64), (1.0, 0), (0, "1")]:
    with raises(ValueError, match="a PCG32's seed and sequence are ints from 0 to 2**64 - 1"):
        Stores.PCG32(seed, sequence)

## RND-02 · A split stream is determined by the seed and the key alone

In [ ]:
source = Stores.PCG32(42)
contact = source.split("Contact")
assert (contact.seed, contact.sequence) == (11264215962035836729, 0)  # FNV-1a 64 of the key, from the basis XOR the seed
assert [contact.next_u32() for _ in range(3)] == [2216387846, 723670704, 428412945]
source.next_u32()
assert source.split("Contact").next_u32() == 2216387846  # what was drawn before does not matter
assert Stores.PCG32(42, 7).split("Contact").sequence == 7 and Stores.PCG32(43).split("Contact").seed != contact.seed
assert source.split("Contact").split("0").split("age").seed == contact.split("0").split("age").seed
assert source.split("Ünïcödé ✓").seed != source.split("Unicode").seed  # by the key's UTF-8 bytes
assert source.split("\U0001F600\u00e9\u0800").seed != source.split("").seed  # every width of UTF-8
with raises(ValueError, match="a key must be text without lone surrogates"):
    source.split("\ud800")

## RND-03 · A store is equipped with a source when it is made

In [ ]:
source = Stores.PCG32(7)
assert Proxies.OfStore(random=source).random() is source
Item = S.OfObject.Builder().name("Item").ref().properties(text("name")).create()
bound = Bd.OfStore([(Item, lambda instance=None: None)], random=source)
assert bound.random() is source
with raises(LookupError, match="the store has no random source: give it one when it is made"):
    Proxies.OfStore().random()
assert conformance_problems(Stores.PCG32, Stores.Random) == []